# 08 — Clasificación topológica de huecos H₁

## ¿Qué hacemos aquí?

La posición de cada hueco en el diagrama de persistencia (birth, persistencia) **no es aleatoria**: codifica el tipo de problema de cobertura de salud que representa. Clasificamos los huecos en cuatro **arquetipos topológicos** con significado real para planeación urbana.

## Los cuatro arquetipos

Los umbrales se calculan sobre el conjunto combinado de regiones (para que sean comparables entre estados):

```
┌─────────────────────────────────────────────────────┐
│              birth BAJO         birth ALTO           │
│  pers ALTA   Enclave sin    │   Desierto             │
│              cobertura [1]  │   estructural [2]      │
├─────────────────────────────┼──────────────────────-─┤
│  pers BAJA   Micro-brecha   │   Vacío periférico     │
│              [4]            │   [3]                  │
└─────────────────────────────────────────────────────┘
```

| Arquetipo | Significado real | Acción |
|-----------|-----------------|--------|
| **[1] Enclave sin cobertura** | Rodeado de servicios pero con vacío interior | Abrir consultorio dentro del hueco |
| **[2] Desierto estructural** | Zona rural sin infraestructura en radio amplio | Inversión estructural desde cero |
| **[3] Vacío periférico** | Transición urbano-rural, servicios espaciados | Ampliar cobertura de servicios existentes |
| **[4] Micro-brecha** | Ruido urbano normal, huecos pequeños | Sin acción urgente |

In [ ]:
import sys
from pathlib import Path
_ROOT = Path("..").resolve()
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from lib import data, tda, clasificar, config

## Cálculo de huecos y clasificación

Los umbrales de birth y persistencia se calculan sobre la **mediana del conjunto combinado** — así la clasificación es objetiva y comparable entre cualquier par de regiones o estados.

In [ ]:
REGIONES = ["CDMX", "EDOMEX"]
MIN_PERS = 200.0

print("Calculando huecos H₁...")
diags = {}
for region in REGIONES:
    df = pd.read_parquet(config.INTERMEDIOS_DIR / f"salud_{region}.parquet")
    P  = data.puntos(df)
    st = tda.alpha_complex(P)
    dr = tda.a_radio(tda.persistencia(st))
    d  = dr[1]
    mask = np.isfinite(d[:, 1]) & ((d[:, 1] - d[:, 0]) >= MIN_PERS)
    diags[region] = d[mask]
    print(f"  {region}: {mask.sum()} huecos")

umb_birth, umb_pers = clasificar.umbrales_conjuntos(diags)
print(f"\nUmbrales: birth={umb_birth/1000:.2f} km | persistencia={umb_pers/1000:.2f} km | crítico=3 km")

dfs_clas = {}
for region in REGIONES:
    df_c = clasificar.clasificar_huecos(diags[region], umb_birth, umb_pers, umbral_critico=3000.0)
    dfs_clas[region] = df_c
    print(f"\n{region}:")
    for _, row in clasificar.resumen_arquetipos(df_c, region).iterrows():
        if row["n"] > 0:
            print(f"  [{row['prioridad']}] {row['arquetipo']:25s} n={row['n']:3d}  "
                  f"pers_media={row['pers_media_km']:.2f} km  pers_max={row['pers_max_km']:.2f} km")

## Panel de clasificación

El panel tiene 5 subgráficas:

1. **Diagrama de persistencia CDMX** coloreado por arquetipo — cada punto es un hueco, anotados los 3 peores.
2. **Diagrama de persistencia EDOMEX** ídem — notar los puntos morados (desiertos) desplazados a la derecha.
3. **Barras de composición** — % de huecos de cada tipo por región. La diferencia en "Desierto estructural" es la más relevante: EDOMEX 40% vs CDMX 8%.
4. **Curva de Betti β₁** — número de huecos activos a cada radio. El pico de EDOMEX es más tardío (huecos activos en radios mayores).
5. **Guía de arquetipos** con conteos por región.

In [ ]:
# Ejecutar el script completo (reutiliza toda la lógica del .py)
exec(open(str(_ROOT / "notebooks" / "08_clasificacion_huecos.py")).read())

## Resultados clave

| Arquetipo | CDMX | EDOMEX | Diferencia |
|-----------|------|--------|------------|
| [1] Enclave sin cobertura | 23 | 27 | Similar — problema de distribución en ambos |
| [2] Desierto estructural | **6** | **84** | EDOMEX tiene **14×** más — problema estructural |
| [3] Vacío periférico | 6 | 44 | EDOMEX más disperso geográficamente |
| [4] Micro-brecha | 36 | 54 | Diferencia moderada |

**El dato más importante:** los 84 desiertos estructurales de EDOMEX tienen una persistencia máxima de **11.81 km** (vs 1.46 km en CDMX). Eso equivale a ~6 km de radio sin ningún servicio de salud — zonas que requieren inversión desde cero y no solo redistribución.